1) Standardized CSV Loader (Robust)

In [9]:
import pandas as pd
from pathlib import Path

TABLE_DIR = Path("results/tables")

def load_quant():
    df = pd.read_csv(TABLE_DIR / "quant_results.csv")

    df = df.rename(columns={
        "quant": "bitwidth",
        "latency_ms": "latency_ms_per_token"
    })

    return df[["bitwidth", "seq_len", "latency_ms_per_token"]]

def load_window():
    df = pd.read_csv(TABLE_DIR / "window_results.csv")

    df = df.rename(columns={
        "window": "window_size",
        "latency_ms": "latency_ms_per_token"
    })

    return df[["window_size", "seq_len", "latency_ms_per_token"]]

quant_df = load_quant()
window_df = load_window()

print("CSVs loaded successfully.")

FileNotFoundError: [Errno 2] No such file or directory: 'results/tables/quant_results.csv'

2) Latency Overlay (Model vs Measured)


In [ ]:
import numpy as np

# Hardware constants (use your actual values)
PEAK_FLOPS = 312e12
PEAK_BW = 1.6e12
E_BYTE_HBM = 5e-12
E_FLOP = 1e-12

def latency_model(seq_len, bitwidth):
    # Replace this body with your exact latency formula
    bytes_per_token = seq_len * bitwidth / 8
    flops_per_token = seq_len * 1e6  # placeholder scaling

    mem_time = bytes_per_token / PEAK_BW
    compute_time = flops_per_token / PEAK_FLOPS

    return max(mem_time, compute_time) * 1000  # ms

def energy_model(seq_len, bitwidth):
    bytes_per_token = seq_len * bitwidth / 8
    flops_per_token = seq_len * 1e6

    energy = bytes_per_token * E_BYTE_HBM + flops_per_token * E_FLOP
    return energy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- Ensure correct numeric types ---
quant_df["seq_len"] = pd.to_numeric(quant_df["seq_len"])

# Convert bitwidth labels to integers if needed
if quant_df["bitwidth"].dtype == object:
    quant_df["bitwidth"] = quant_df["bitwidth"].replace({
        "FP16": 16,
        "INT8": 8,
        "INT4": 4
    })

quant_df["bitwidth"] = pd.to_numeric(quant_df["bitwidth"])

# --- Plot ---
plt.figure(figsize=(6,4))

for bit in sorted(quant_df["bitwidth"].unique()):
    df_sub = quant_df[quant_df["bitwidth"] == bit].sort_values("seq_len")

    T = df_sub["seq_len"].values.astype(float)
    measured = df_sub["latency_ms_per_token"].values.astype(float)
    predicted = np.array([latency_model(t, bit) for t in T])

    plt.plot(T, predicted, linewidth=2, label=f"Model {int(bit)}-bit")
    plt.scatter(T, measured, s=40)

plt.xscale("log")
plt.xlabel("Sequence Length")
plt.ylabel("Latency (ms / token)")
plt.legend()
plt.tight_layout()

Path("results/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("results/figures/latency_overlay.png", dpi=300)

plt.show()

4) Error Metrics (MAE, Relative MAE, RMSE)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

results = []

for bit in sorted(quant_df["bitwidth"].unique()):
    df_sub = quant_df[quant_df["bitwidth"] == bit].sort_values("seq_len")

    T = df_sub["seq_len"].values.astype(float)
    measured = df_sub["latency_ms_per_token"].values.astype(float)
    predicted = np.array([latency_model(t, bit) for t in T])

    mae = np.mean(np.abs(predicted - measured))
    rmse = np.sqrt(np.mean((predicted - measured)**2))
    rel_mae = mae / np.mean(measured) * 100

    results.append({
        "bitwidth": int(bit),
        "MAE_ms": mae,
        "RMSE_ms": rmse,
        "Rel_MAE_percent": rel_mae
    })

error_df = pd.DataFrame(results).sort_values("bitwidth")

print(error_df)

Path("results/tables").mkdir(parents=True, exist_ok=True)
error_df.to_csv("results/tables/model_latency_error.csv", index=False)

5) Energy vs Sequence Length (Model)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.figure(figsize=(6,4))

unique_T = sorted(quant_df["seq_len"].unique())

for bit in sorted(quant_df["bitwidth"].unique()):
    energies = [energy_model(t, bit) for t in unique_T]
    plt.plot(unique_T, energies, linewidth=2, label=f"{int(bit)}-bit")

plt.xscale("log")
plt.xlabel("Sequence Length")
plt.ylabel("Energy (J / token)")
plt.legend()
plt.tight_layout()

Path("results/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("results/figures/energy_vs_seq_model.png", dpi=300)

plt.show()

6) Finally summary table

In [ ]:
import pandas as pd
from pathlib import Path

T_ref = 4096  # choose representative context length

rows = []

for bit in sorted(quant_df["bitwidth"].unique()):
    latency_pred = latency_model(T_ref, bit)
    energy_pred = energy_model(T_ref, bit)

    rows.append({
        "Bitwidth": int(bit),
        "Latency_ms_per_token": latency_pred,
        "Energy_J_per_token": energy_pred
    })

summary_df = pd.DataFrame(rows).sort_values("Bitwidth")

print(summary_df)

Path("results/tables").mkdir(parents=True, exist_ok=True)
summary_df.to_csv("results/tables/final_summary_table.csv", index=False)

summary_df.to_latex(
    "paper/final_summary_table.tex",
    index=False,
    float_format="%.4f"
)